# 06 · Anticipation strategy: long predicted additions, short predicted deletions

This notebook builds the project's trading strategy. Notebook 02 shows that the **post-event** index effect is absent in 2010–2022, so a strategy that trades after the effective date has nothing left to earn. The price pressure has not disappeared, but it now happens **before** the event. This notebook trades that pre-event window with U.S. common stocks only: long positions and short sales of individual securities, with no derivatives and no ETFs.

**Design, fixed before computing any strategy return**

| Element | Choice |
|---|---|
| Long leg | Eligible non-members with the highest predicted probability of being added within 3 months (top 20, equal-weighted) |
| Short leg | S&P 500 members with the highest predicted probability of a discretionary deletion within 3 months (top 20, equal-weighted) |
| Spread | Long leg minus short leg, dollar-neutral, no hedge instrument |
| Formation | Each month-end `t`, using data available at `t`. Positions are held in month `t+1` |
| Exit | If the security's index trade date falls in the holding month, the position is closed at that day's close and the proceeds earn the risk-free rate for the rest of the month |
| Models | Logits estimated once on formation months 1976-01 to 2009-09, whose 3-month labels are known by December 2009. Coefficients stay fixed afterwards |
| Out-of-sample test | Return months 2010-01 to 2022-12 (headline). 1976–2009 returns are in sample and reported only as description |
| Alpha | Fama–French five factors plus momentum, Newey–West with 12 lags on the monthly calendar |
| Costs | Quoted half-spread at formation (or 0.2/0.5/1.0% per side) on actual turnover, including exits; borrow fee of 0.5/1/2% a year on the short leg |

**Why exit at the index trade date.** Section 1 decomposes the event-study CARs by sub-window. In the training period, deletions fall by about 14% in the 20 sessions up to the index trade date and then rebound by about 8% in the next 20 sessions. Additions rise into the trade date and are flat afterwards. Closing at the trade-date close captures the pressure and avoids the rebound. The 3-month horizon and the top-20 list size were set in advance and were not tuned.

**No look-ahead and no ex-post filters.** Candidate universes, features and the trade-date exit use only information available at the time. The deletion label is a *discretionary* deletion (not followed by a delisting within 21 sessions), because those are the deletions that come with selling pressure. The label is only the training target. The short portfolio holds every selected member, including those later acquired, whose takeover premium is a loss for the short.

**Disclosure.** The long-leg inputs (size rank, liquidity and the S&P earnings rule, model E) were chosen in notebook 04, which had already seen prediction results for 2013–2022. That choice is therefore not fully independent of the test period. The short-leg inputs are new and were set before any result in this notebook.

In [ ]:
from pathlib import Path

from IPython.utils.capture import capture_output
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score

inventory_path = Path("01_inventory.ipynb")
if not inventory_path.is_file():
    inventory_path = Path("notebooks") / "01_inventory.ipynb"
with capture_output(stdout=False, stderr=False):
    get_ipython().run_line_magic("run", f'"{inventory_path.resolve().as_posix()}"')

top_n = 20
label_horizon = 3
newey_west_lags = 12
earnings_lag_months = 4
fallback_half_spread = 0.005
first_formation, last_formation = pd.Period("1976-01", "M"), pd.Period("2022-11", "M")
train_formation_end = pd.Period("2009-09", "M")      # its 3-month labels are known by December 2009
label_end = history_end.to_period("M")
test_months = pd.period_range("2010-01", "2022-12", freq="M")
train_months = pd.period_range("1976-02", "2009-12", freq="M")
periods = {
    "2010–2022 (out of sample)": test_months,
    "1976–2009 (in sample)": train_months,
    "1976–1989 (in sample)": pd.period_range("1976-02", "1989-12", freq="M"),
    "1990–1999 (in sample)": pd.period_range("1990-01", "1999-12", freq="M"),
    "2000–2009 (in sample)": pd.period_range("2000-01", "2009-12", freq="M"),
}
headline = "2010–2022 (out of sample)"

# Every reported estimate with a t-statistic is logged to count specifications.
specification_log = []


def log_specifications(table_name, t_values):
    t_values = pd.Series(t_values).dropna()
    specification_log.append({
        "table": table_name, "estimates": len(t_values),
        "|t| > 2": int(t_values.abs().gt(2).sum()), "|t| > 3": int(t_values.abs().gt(3).sum()),
    })


main_additions = additions_register.loc[additions_register["main_sample"]]
main_deletions = deletions_register.loc[deletions_register["main_sample"]]
display(pd.Series({
    "main additions, 1976–2022": len(main_additions),
    "main (discretionary) deletions, 1976–2022": len(main_deletions),
}, name="events"))

## 1 · When does the price pressure happen?

Daily CARs against the S&P 500 return, centred on the index trade date (day 0), computed as in notebook 02: complete windows only, abnormal return 0 after a delisting, t-statistics clustered by effective month. The training period (1976–2009) fixes the exit rule. The 2010–2022 column is shown for completeness and was not used for any design choice.

In [ ]:
expected_daily_rows = 92_586_611  # data rows in the daily CSV export, as checked in notebook 02
daily_dataset = ds.dataset(data_dir / "parquet" / "crsp_daily", format="parquet")
if daily_dataset.count_rows() != expected_daily_rows:
    raise ValueError("The Parquet copy of the daily file does not match the CSV row count.")
daily_columns = ["permno", "dlycaldt", "dlydelflg", "dlyret", "dlybid", "dlyask"]


def read_daily(permnos):
    frame = daily_dataset.to_table(
        columns=daily_columns, filter=ds.field("permno").isin(sorted(permnos))
    ).to_pandas()
    frame["date"] = pd.to_datetime(frame.pop("dlycaldt"))
    frame = frame.drop_duplicates()
    conflicting = frame.duplicated(["permno", "date"], keep=False)
    frame.loc[conflicting, ["dlyret", "dlybid", "dlyask"]] = np.nan
    frame = frame.drop_duplicates(["permno", "date"])
    frame["session"] = frame["date"].map(session_of)
    return frame.dropna(subset=["session"]).astype({"session": "int64"}).sort_values(["permno", "session"], ignore_index=True)


event_permnos = set(pd.concat([additions_register["permno"], deletions_register["permno"]]).dropna().astype(int))
event_daily = read_daily(event_permnos)
benchmark = pd.read_csv(paths["index_daily"], usecols=["dlycaldt", "sprtrn"], parse_dates=["dlycaldt"])
event_daily = event_daily.merge(benchmark.rename(columns={"dlycaldt": "date"}), on="date", how="left")
event_daily["ar"] = event_daily["dlyret"] - event_daily["sprtrn"]
event_daily.loc[event_daily["dlydelflg"].eq("Y") & event_daily["dlyret"].isna(), "ar"] = 0.0
span = event_daily.groupby("permno")["session"].agg(first_session="min", last_session="max")
last_calendar_session = len(trading_calendar) - 1

timing_windows = {
    "[-250,-121]": (-250, -121), "[-120,-61]": (-120, -61), "[-60,-41]": (-60, -41), "[-40,-21]": (-40, -21),
    "[-20,-6]": (-20, -6), "[-5,-1]": (-5, -1), "[0]": (0, 0), "[+1,+20]": (1, 20),
}


def window_cars(register):
    events = register[["permno", "effective_date", "index_trade_date"]].astype({"permno": "int64"}).reset_index(drop=True)
    events["anchor"] = events["index_trade_date"].map(session_of)
    events = events.join(span, on="permno")
    rows = events.reset_index().merge(event_daily[["permno", "session", "ar"]], on="permno")
    rows["relative"] = rows["session"] - rows["anchor"]
    out = events[["effective_date"]].copy()
    for window, (low, high) in timing_windows.items():
        part = rows.loc[rows["relative"].between(low, high)].groupby("index")["ar"].agg(["sum", "count"])
        start, end = events["anchor"] + low, events["anchor"] + high
        expected = (np.minimum(end, events["last_session"]) - start + 1).clip(lower=0)
        complete = (events["first_session"].lt(start) & end.le(last_calendar_session) & expected.gt(0)
                    & part["count"].reindex(events.index).fillna(0).eq(expected))
        out[window] = part["sum"].reindex(events.index).where(complete)
    return out


def clustered_t(values, clusters):
    fit = sm.OLS(np.asarray(values, float), np.ones(len(values))).fit(
        cov_type="cluster", cov_kwds={"groups": pd.factorize(clusters)[0]})
    return fit.tvalues[0]


timing_groups = {
    "additions (main)": main_additions,
    "deletions (main, discretionary)": main_deletions,
    "deletions followed by delisting (excluded from the label)": deletions_register.loc[
        deletions_register["sample"].eq("core") & deletions_register["permno"].notna()
        & deletions_register["flag_delisting_related"]],
}
timing_rows, timing_t = [], []
for group, register in timing_groups.items():
    cars = window_cars(register)
    years = cars["effective_date"].dt.year
    for period, selected in [("1976–2009 (training)", years.le(2009)), ("2010–2022", years.ge(2010))]:
        sample = cars.loc[selected]
        for window in timing_windows:
            values = sample[window].dropna()
            t_value = clustered_t(values, sample.loc[values.index, "effective_date"].dt.to_period("M"))
            timing_t.append(t_value)
            timing_rows.append({"group": group, "period": period, "window": window, "events": len(values),
                                "mean_car_pct": 100 * values.mean(), "t_stat": t_value})
timing_table = pd.DataFrame(timing_rows)
log_specifications("CAR timing by sub-window", timing_t)
display(timing_table.pivot_table(index="window", columns=["group", "period"], values=["mean_car_pct", "t_stat"], sort=False)
        .swaplevel(0, 1, axis=1).swaplevel(1, 2, axis=1).sort_index(axis=1, level=[0, 1], sort_remaining=False).round(2))

## 2 · Candidate universes and features

**Long universe.** Eligible non-members at month-end `t` (U.S. ordinary common stock on NYSE, NYSE American or Nasdaq, as in notebook 01), excluding firms with another share class already in the index (as in notebook 04).

**Short universe.** S&P 500 members at month-end `t` with a positive market capitalization. Size is measured at the firm level (sum of the firm's member share classes), because S&P judges the company and not one share class.

**Features**, all known at `t`:

- long leg (model E of notebook 04): log size rank among candidates (1 = largest), log dollar volume, dollar volume over market cap, and the S&P earnings rule (positive latest quarter and positive four-quarter sum) with a four-month reporting lag;
- short leg: log size rank among members counted from the bottom (1 = smallest), the same two liquidity measures, the same earnings rule, and 12-1 month momentum, because discretionary deletions follow large price declines.

Liquidity and momentum are winsorized at 1% and 99% and standardized within each month and universe. A security whose index trade date is the formation date itself is not eligible for selection: the index trade happens at that close, so a position opened then would only hold the post-event return.

In [ ]:
firm_ids = pd.read_csv(
    paths["crsp_monthly"], usecols=["PERMNO", "PERMCO", "MthCalDt"], parse_dates=["MthCalDt"]
).rename(columns={"PERMNO": "permno", "PERMCO": "permco", "MthCalDt": "mthcaldt"}).drop_duplicates()
if firm_ids.duplicated(["permno", "mthcaldt"]).any():
    raise ValueError("A security has more than one PERMCO in a month.")
panel = monthly_panel[["permno", "mthcaldt", "mthcap", "mthprc", "mthret", "mthvol", "mthdelflg", "in_sp500", "eligible"]].merge(
    firm_ids, on=["permno", "mthcaldt"], how="left", validate="one_to_one")
panel = panel.sort_values(["permno", "mthcaldt"], ignore_index=True)
panel["month"] = panel["mthcaldt"].dt.to_period("M")

security = panel["permno"]
month_number = panel["mthcaldt"].dt.year * 12 + panel["mthcaldt"].dt.month
with np.errstate(divide="ignore", invalid="ignore"):
    log_return = np.log1p(panel["mthret"])


def months_since(lag):
    return month_number - month_number.groupby(security).shift(lag)


def trailing_sum(values, window):
    rolled = values.groupby(security).rolling(window, min_periods=window).sum().droplevel(0)
    return rolled.where(months_since(window - 1).eq(window - 1))


dollar_volume = panel["mthvol"] * panel["mthprc"].abs()
panel["log_dollar_volume"] = np.log(dollar_volume.where(dollar_volume.gt(0)))
panel["dollar_volume_to_mktcap"] = dollar_volume / (panel["mthcap"] * 1000)  # MthCap is in $ thousands
panel["mom_12_1"] = np.expm1(trailing_sum(log_return, 11).groupby(security).shift(1).where(months_since(1).eq(1)))

member_firm_cap = panel.loc[panel["in_sp500"] & panel["mthcap"].gt(0)].groupby(["permco", "month"])["mthcap"].sum()
panel["firm_member_cap"] = member_firm_cap.reindex(pd.MultiIndex.from_frame(panel[["permco", "month"]])).to_numpy()
in_window = panel["month"].between(first_formation, last_formation)

long_rows = panel.loc[in_window & ~panel["in_sp500"] & panel["eligible"] & panel["firm_member_cap"].isna()].copy()
long_rows["size_rank"] = long_rows.groupby("month")["mthcap"].rank(ascending=False, method="min")
long_rows["log_size_rank"] = np.log(long_rows["size_rank"])

short_rows = panel.loc[in_window & panel["in_sp500"] & panel["mthcap"].gt(0)].copy()
short_rows["bottom_rank"] = short_rows.groupby("month")["firm_member_cap"].rank(ascending=True, method="min")
short_rows["log_bottom_rank"] = np.log(short_rows["bottom_rank"])


def standardize(rows, columns):
    by_month = rows.groupby("month")[columns]
    clipped = rows[columns].clip(lower=by_month.transform(lambda c: c.quantile(0.01)),
                                 upper=by_month.transform(lambda c: c.quantile(0.99)), axis=0)
    grouped = clipped.groupby(rows["month"])
    rows[columns] = (clipped - grouped.transform("mean")) / grouped.transform("std")


standardize(long_rows, ["log_dollar_volume", "dollar_volume_to_mktcap", "mom_12_1"])
standardize(short_rows, ["log_dollar_volume", "dollar_volume_to_mktcap", "mom_12_1"])
display(pd.DataFrame({
    "long universe": long_rows.groupby(long_rows["month"].dt.year // 10 * 10).size()
    / long_rows.groupby(long_rows["month"].dt.year // 10 * 10)["month"].nunique(),
    "short universe": short_rows.groupby(short_rows["month"].dt.year // 10 * 10).size()
    / short_rows.groupby(short_rows["month"].dt.year // 10 * 10)["month"].nunique(),
}).round(0).rename_axis("decade: mean securities per month"))

In [ ]:
# S&P earnings rule: positive latest quarter and positive sum of the last four, known four months after quarter-end.
quarterly = pd.read_csv(
    paths["comp_quarterly"], usecols=["gvkey", "datadate", "ibq"], parse_dates=["datadate"]
).dropna(subset=["ibq"]).drop_duplicates(["gvkey", "datadate", "ibq"])
conflicts = quarterly.duplicated(["gvkey", "datadate"], keep=False)
quarterly = quarterly.loc[~conflicts].sort_values(["gvkey", "datadate"], ignore_index=True)
quarterly["quarter_month"] = (quarterly["datadate"].dt.year * 12 + quarterly["datadate"].dt.month).astype("int64")
by_firm = quarterly.groupby("gvkey")
quarterly["earnings_4q"] = by_firm["ibq"].rolling(4, min_periods=4).sum().droplevel(0)
quarterly["consecutive_4q"] = quarterly["quarter_month"] - by_firm["quarter_month"].shift(3) == 9
quarterly["passes_rule"] = quarterly["ibq"].gt(0) & quarterly["earnings_4q"].gt(0)

links = pd.read_csv(paths["ccm_link"], usecols=["gvkey", "LPERMNO", "LINKPRIM", "LINKDT", "LINKENDDT"])
links = links.loc[links["LINKPRIM"].isin(["P", "C"])].rename(columns={"LPERMNO": "permno"}).copy()
links["link_start"] = pd.to_datetime(links["LINKDT"])
links["link_end"] = pd.to_datetime(links["LINKENDDT"].replace("E", "2099-12-31"))


def earnings_status(rows, lag_months=earnings_lag_months, max_age_months=12):
    keys = rows[["permno", "mthcaldt"]].drop_duplicates()
    linked = keys.merge(links[["permno", "gvkey", "link_start", "link_end"]], on="permno")
    linked = linked.loc[linked["mthcaldt"].between(linked["link_start"], linked["link_end"]), ["permno", "mthcaldt", "gvkey"]]
    linked = linked.loc[~linked.duplicated(["permno", "mthcaldt"], keep=False)]
    linked["formation_month"] = (linked["mthcaldt"].dt.year * 12 + linked["mthcaldt"].dt.month).astype("int64")
    known = quarterly.assign(known_month=quarterly["quarter_month"] + lag_months).sort_values("known_month")
    matched = pd.merge_asof(
        linked.sort_values("formation_month"), known[["gvkey", "known_month", "quarter_month", "consecutive_4q", "passes_rule"]],
        left_on="formation_month", right_on="known_month", by="gvkey", direction="backward")
    if (matched["formation_month"] - matched["quarter_month"]).lt(lag_months).any():
        raise ValueError("An earnings quarter was used before its assumed availability date.")
    usable = matched["consecutive_4q"].eq(True) & (matched["formation_month"] - matched["quarter_month"]).le(max_age_months)
    matched["status"] = np.where(usable, np.where(matched["passes_rule"].eq(True), "pass", "fail"), "unknown")
    status = rows[["permno", "mthcaldt"]].merge(matched[["permno", "mthcaldt", "status"]], on=["permno", "mthcaldt"], how="left")
    return status["status"].fillna("unknown").to_numpy()


for rows in (long_rows, short_rows):
    status = earnings_status(rows)
    rows["earnings_pass"] = (status == "pass").astype(int)
    rows["earnings_unknown"] = (status == "unknown").astype(int)
display(pd.DataFrame({
    "long universe": [long_rows["earnings_pass"].mean(), long_rows["earnings_unknown"].mean()],
    "short universe": [short_rows["earnings_pass"].mean(), short_rows["earnings_unknown"].mean()],
}, index=["share passing", "share unknown"]).round(3))

## 3 · Labels and the trade-date calendar

`y` equals 1 when the security's next event of the leg's type takes effect within the next three months. Additions use the main additions, deletions the main (discretionary) deletions. Labels whose three-month horizon passes December 2022 are missing. A separate table records **every** register event of each type with its index trade date, used by the exit rule, so the exit also applies to deletions caused by mergers.

In [ ]:
def add_labels(rows, targets):
    targets = targets[["permno", "effective_date"]].dropna().astype({"permno": "int64"}).sort_values("effective_date")
    rows = pd.merge_asof(rows.sort_values("mthcaldt"), targets, left_on="mthcaldt", right_on="effective_date",
                         by="permno", direction="forward", allow_exact_matches=False)
    rows["lead_months"] = (rows["effective_date"].dt.to_period("M") - rows["month"]).apply(
        lambda offset: offset.n if pd.notna(offset) else np.nan)
    rows["y"] = rows["lead_months"].le(label_horizon).astype(float).where(rows["month"] + label_horizon <= label_end)
    return rows.rename(columns={"effective_date": "next_effective"}).sort_values(["month", "permno"], ignore_index=True)


long_rows = add_labels(long_rows, main_additions)
short_rows = add_labels(short_rows, main_deletions)

exit_dates = {
    "long": additions_register[["permno", "index_trade_date"]].dropna().astype({"permno": "int64"}),
    "short": deletions_register[["permno", "index_trade_date"]].dropna().astype({"permno": "int64"}),
}
for leg, rows in [("long", long_rows), ("short", short_rows)]:
    traded_now = rows[["permno", "mthcaldt"]].merge(
        exit_dates[leg].rename(columns={"index_trade_date": "mthcaldt"}), on=["permno", "mthcaldt"], how="left", indicator=True)
    rows["trades_at_formation"] = traded_now["_merge"].eq("both").to_numpy()

display(pd.DataFrame({
    leg: {
        "rows": len(rows), "labelled rows": int(rows["y"].notna().sum()), "positive rows": int(rows["y"].sum()),
        "base rate": rows["y"].mean(),
        "events reachable (labelled positive at least once)": rows.loc[rows["y"].eq(1), ["permno", "next_effective"]].drop_duplicates().shape[0],
        "rows excluded: index trade at the formation close": int(rows["trades_at_formation"].sum()),
    } for leg, rows in [("long", long_rows), ("short", short_rows)]
}).round(4))

## 4 · Logits estimated on 1976–2009

Each model is fitted once on formation months January 1976 to September 2009, with standard errors clustered by security and by month. All later months, including the whole test period, are scored with these fixed coefficients.

In [ ]:
model_inputs = {
    "long": ["log_size_rank", "log_dollar_volume", "dollar_volume_to_mktcap", "earnings_pass", "earnings_unknown"],
    "short": ["log_bottom_rank", "log_dollar_volume", "dollar_volume_to_mktcap", "earnings_pass", "earnings_unknown", "mom_12_1"],
}
leg_rows = {"long": long_rows, "short": short_rows}
fits = {}
for leg, rows in leg_rows.items():
    train = rows.loc[rows["month"].le(train_formation_end) & rows["y"].notna()].dropna(subset=model_inputs[leg])
    groups = np.column_stack([pd.factorize(train["permno"])[0], pd.factorize(train["month"])[0]])
    fits[leg] = sm.Logit(train["y"], sm.add_constant(train[model_inputs[leg]])).fit(
        disp=False, maxiter=200, cov_type="cluster", cov_kwds={"groups": groups})
    if not fits[leg].mle_retvals["converged"]:
        raise RuntimeError(f"The {leg} logit did not converge.")
    scorable = rows[model_inputs[leg]].notna().all(axis=1)
    rows["score"] = np.nan
    rows.loc[scorable, "score"] = fits[leg].predict(sm.add_constant(rows.loc[scorable, model_inputs[leg]], has_constant="add"))

display(pd.concat({
    leg: pd.DataFrame({"coefficient": fit.params, "clustered_se": fit.bse, "t": fit.tvalues}) for leg, fit in fits.items()
}).round(3))
display(pd.DataFrame({
    leg: {"rows": int(fit.nobs), "positive rows": int(fit.model.endog.sum()), "pseudo R2": round(fit.prsquared, 4)}
    for leg, fit in fits.items()
}).T)

## 5 · Prediction quality

Precision at 20 is the share of the monthly top 20 whose event takes effect within three months. *Maximum attainable* caps it by the number of positives in each month, and *base rate* is the outcome of random selection. The test rows are formation months December 2009 to September 2022, the last month with a complete label.

In [ ]:
def select_top(rows, score="score", n=top_n):
    usable = rows.loc[~rows["trades_at_formation"]].dropna(subset=[score])
    return (usable.sort_values(["month", score, "permno"], ascending=[True, False, True], kind="stable")
            .groupby("month").head(n))


def prediction_metrics(rows):
    rows = rows.loc[rows["y"].notna() & rows["score"].notna() & ~rows["trades_at_formation"]]
    selected = select_top(rows)
    monthly_precision = selected.groupby("month")["y"].mean()
    events_all = rows.loc[rows["y"].eq(1), ["permno", "next_effective"]].drop_duplicates()
    events_hit = selected.loc[selected["y"].eq(1), ["permno", "next_effective"]].drop_duplicates()
    return {
        "formation months": rows["month"].nunique(), "rows": len(rows), "positive rows": int(rows["y"].sum()),
        "base rate": rows["y"].mean(), "precision at 20": monthly_precision.mean(),
        "maximum attainable": rows.groupby("month")["y"].sum().clip(upper=top_n).div(top_n).mean(),
        "event recall at 20": len(events_hit) / len(events_all), "AUC (pooled)": roc_auc_score(rows["y"], rows["score"]),
    }


prediction = {}
for leg, rows in leg_rows.items():
    prediction[(leg, "1976-01 to 2009-09 (in sample)")] = prediction_metrics(rows.loc[rows["month"].le(train_formation_end)])
    prediction[(leg, "2009-12 to 2022-09 (out of sample)")] = prediction_metrics(rows.loc[rows["month"].ge(pd.Period("2009-12", "M"))])
display(pd.DataFrame(prediction).round(4))

## 6 · Daily data for exits and quotes

The exit rule needs daily returns of the selected securities in their holding month, and the cost model needs the quoted spread at the formation month-end. The cell reads the daily file only for the selected security-months of every portfolio variant used below.

In [ ]:
variants = {
    "main": {"long": select_top(long_rows), "short": select_top(short_rows)},
    "top 10": {"long": select_top(long_rows, n=10), "short": select_top(short_rows, n=10)},
    "top 40": {"long": select_top(long_rows, n=40), "short": select_top(short_rows, n=40)},
    # Naive benchmarks without a model: the 20 largest candidates and the 20 smallest members.
    "size only": {"long": select_top(long_rows.assign(size_score=-long_rows["size_rank"]), "size_score"),
                  "short": select_top(short_rows.assign(size_score=-short_rows["bottom_rank"]), "size_score")},
}
needed = pd.concat([
    selected[["permno", "month"]].assign(month=selected["month"] + offset)
    for legs in variants.values() for selected in legs.values() for offset in (0, 1)
]).drop_duplicates()
candidate_daily = read_daily(set(needed["permno"]))
candidate_daily["month"] = candidate_daily["date"].dt.to_period("M")
candidate_daily = candidate_daily.merge(needed, on=["permno", "month"])

quotes = candidate_daily.loc[candidate_daily["dlybid"].gt(0) & candidate_daily["dlyask"].gt(candidate_daily["dlybid"])].copy()
quotes["relative_spread"] = (quotes["dlyask"] - quotes["dlybid"]) / ((quotes["dlyask"] + quotes["dlybid"]) / 2)
quotes = quotes.loc[quotes["relative_spread"].lt(0.20)]
half_spread = (quotes.sort_values("date").groupby(["permno", "month"])["relative_spread"].last() / 2).rename("half_spread")

monthly_factors = read_monthly_factors(
    data_dir / "F-F_Research_Data_5_Factors_2x3_CSV.zip", ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]
).join(read_monthly_factors(data_dir / "F-F_Momentum_Factor_CSV.zip", ["Mom"]), validate="one_to_one")
factor_columns = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom"]
market = pd.read_csv(paths["index_monthly"], usecols=["mthcaldt", "vwretd"], parse_dates=["mthcaldt"])
market = market.assign(month=market["mthcaldt"].dt.to_period("M")).set_index("month")["vwretd"]
sessions = pd.Series(trading_calendar, index=trading_calendar).to_frame("date")
sessions["month"] = sessions["date"].dt.to_period("M")
sessions_in_month = sessions.groupby("month").size()
monthly_returns = panel.set_index(["permno", "month"])[["mthret", "mthcap", "mthdelflg"]]
delisting_month = delisting_info.dropna(subset=["delistingdt"]).assign(
    delisting_month=lambda f: f["delistingdt"].dt.to_period("M")).set_index("permno")["delisting_month"]
display(pd.Series({"daily rows read for candidates": len(candidate_daily),
                   "security-months with a quoted spread": len(half_spread)}, name="daily data"))

## 7 · Portfolio returns and costs

For each leg, a selected security earns its CRSP monthly return in the holding month, which includes any delisting return. If its index trade date falls in the holding month, it earns instead its compounded daily return through that day's close, and then the risk-free rate pro rata for the remaining sessions. A position without any return is valued at 0 and counted.

**Costs.** At each formation, the leg trades the difference between the new equal weights and the drifted weights of the previous holdings. Positions closed at the trade date are sold during the month and repurchased only if selected again, and both trades are charged. Positions that end in a delisting return cash at no cost. Each trade pays the quoted half-spread at the formation month-end. A missing quote takes that month's median among traded securities, or 0.5% when no quote exists. The borrow fee is charged on the full short leg.

In [ ]:
def leg_positions(selected, leg, exit_rule=True, weighting="equal"):
    positions = selected[["permno", "month", "y", "lead_months", "next_effective"]].rename(columns={"month": "formation_month"})
    positions["month"] = positions["formation_month"] + 1
    keys = pd.MultiIndex.from_frame(positions[["permno", "month"]])
    positions["ret"] = monthly_returns["mthret"].reindex(keys).to_numpy()
    positions["mthdelflg"] = monthly_returns["mthdelflg"].reindex(keys).to_numpy()
    positions["formation_cap"] = monthly_returns["mthcap"].reindex(
        pd.MultiIndex.from_arrays([positions["permno"], positions["formation_month"]])).to_numpy()
    positions["delists"] = positions["permno"].map(delisting_month).eq(positions["month"])
    trades = exit_dates[leg].assign(month=lambda f: f["index_trade_date"].dt.to_period("M"))
    positions = positions.merge(trades.drop_duplicates(["permno", "month"]), on=["permno", "month"], how="left")
    positions["exits"] = positions["index_trade_date"].notna() & exit_rule
    if positions["exits"].any():
        exits = positions.loc[positions["exits"], ["permno", "month", "index_trade_date"]].reset_index()
        days = exits.merge(candidate_daily[["permno", "month", "date", "dlyret"]], on=["permno", "month"])
        days = days.loc[days["date"].le(days["index_trade_date"])]
        to_exit = days.groupby("index").agg(growth=("dlyret", lambda r: np.prod(1 + r.fillna(0.0))), days=("date", "size"))
        after = exits.set_index("index").apply(
            lambda r: int(((sessions["month"] == r["month"]) & (sessions["date"] > r["index_trade_date"])).sum()), axis=1)
        rf = monthly_factors["RF"].reindex(positions.loc[after.index, "month"]).to_numpy()
        exit_ret = to_exit["growth"].reindex(after.index) * (1 + rf * after / sessions_in_month.reindex(
            positions.loc[after.index, "month"]).to_numpy()) - 1
        no_daily = exit_ret.isna()
        positions.loc[exit_ret.index[~no_daily], "ret"] = exit_ret[~no_daily]
        positions.loc[exit_ret.index[no_daily], "exits"] = False
    positions["missing_return"] = positions["ret"].isna()
    positions["ret"] = positions["ret"].fillna(0.0)
    if weighting == "value":
        positions = positions.loc[positions["formation_cap"].gt(0)].copy()
        positions["weight"] = positions["formation_cap"] / positions.groupby("month")["formation_cap"].transform("sum")
    else:
        positions["weight"] = 1 / positions.groupby("month")["permno"].transform("size")
    return positions


def leg_costs(positions, months):
    series = positions.assign(w_ret=positions["weight"] * positions["ret"]).groupby("month")["w_ret"].sum()
    previous = positions.join(series.rename("leg_ret"), on="month")
    previous["drifted"] = np.where(previous["exits"] | previous["delists"], 0.0,
                                   previous["weight"] * (1 + previous["ret"]) / (1 + previous["leg_ret"]))
    previous["month"] = previous["month"] + 1
    rebalance = positions[["permno", "month", "weight"]].merge(previous[["permno", "month", "drifted"]], on=["permno", "month"], how="outer")
    rebalance["traded"] = (rebalance["weight"].fillna(0) - rebalance["drifted"].fillna(0)).abs()
    exit_sales = positions.loc[positions["exits"], ["permno", "month"]].assign(
        traded=positions.loc[positions["exits"], "weight"] * (1 + positions.loc[positions["exits"], "ret"]))
    trades = pd.concat([rebalance[["permno", "month", "traded"]], exit_sales], ignore_index=True)
    trades = trades.loc[trades["month"].isin(months) & trades["traded"].gt(0)]
    trades["quote_month"] = trades["month"] - 1
    trades = trades.join(half_spread, on=["permno", "quote_month"])
    trades["quoted"] = trades["half_spread"].notna()
    trades["half_spread"] = trades["half_spread"].fillna(trades.groupby("month")["half_spread"].transform("median")).fillna(fallback_half_spread)
    by_month = trades.assign(cost=trades["traded"] * trades["half_spread"]).groupby("month").agg(
        turnover=("traded", "sum"), spread_cost=("cost", "sum"))
    return by_month.reindex(months).fillna(0.0), trades


all_months = pd.period_range(first_formation + 1, last_formation + 1, freq="M")
cost_scenarios = {"estimated spread": None, "0.2% per side": 0.002, "0.5% per side": 0.005, "1.0% per side": 0.010}


def build_strategy(legs, exit_rule=True, weighting="equal"):
    frame = pd.DataFrame(index=all_months).join(monthly_factors).join(market.rename("market"))
    details = {}
    for leg in ("long", "short"):
        positions = leg_positions(legs[leg], leg, exit_rule, weighting)
        grouped = positions.assign(w_ret=positions["weight"] * positions["ret"]).groupby("month")
        frame[f"{leg}_ret"] = grouped["w_ret"].sum()
        frame[f"{leg}_positions"] = grouped.size()
        costs, trades = leg_costs(positions, all_months)
        frame[f"{leg}_turnover"] = costs["turnover"]
        frame[f"{leg}_cost"] = costs["spread_cost"]
        details[leg] = {"positions": positions, "trades": trades}
    frame["Long leg"] = frame["long_ret"] - frame["RF"]
    frame["Short leg"] = frame["RF"] - frame["short_ret"]
    frame["Long-short spread"] = frame["long_ret"] - frame["short_ret"]
    for scenario, per_side in cost_scenarios.items():
        cost = {leg: frame[f"{leg}_cost"] if per_side is None else frame[f"{leg}_turnover"] * per_side for leg in ("long", "short")}
        frame[f"Long leg | {scenario}"] = frame["Long leg"] - cost["long"]
        frame[f"Short leg | {scenario}"] = frame["Short leg"] - cost["short"]
        frame[f"Long-short spread | {scenario}"] = frame["Long-short spread"] - cost["long"] - cost["short"]
    for fee in (0.005, 0.01, 0.02):
        for strategy in ("Short leg", "Long-short spread"):
            frame[f"{strategy} | estimated spread + {fee:.1%} borrow"] = frame[f"{strategy} | estimated spread"] - fee / 12
    return frame, details


main_frame, main_details = build_strategy(variants["main"])
display(pd.DataFrame({
    leg: {
        "position-months": len(details["positions"]),
        "mean positions per month": main_frame[f"{leg}_positions"].mean(),
        "positions closed at the index trade date": int(details["positions"]["exits"].sum()),
        "positions ending in a delisting": int(details["positions"]["delists"].sum()),
        "position-months without a return (valued at 0)": int(details["positions"]["missing_return"].sum()),
        "mean monthly turnover, 2010–2022": main_frame.loc[test_months, f"{leg}_turnover"].mean(),
        "traded weight with a quote, 2010–2022": np.average(
            details["trades"].loc[details["trades"]["month"].isin(test_months), "quoted"],
            weights=details["trades"].loc[details["trades"]["month"].isin(test_months), "traded"]),
        "weighted mean half-spread, 2010–2022 (%)": 100 * np.average(
            details["trades"].loc[details["trades"]["month"].isin(test_months), "half_spread"],
            weights=details["trades"].loc[details["trades"]["month"].isin(test_months), "traded"]),
    } for leg, details in main_details.items()
}).round(3))

## 8 · Alphas

Monthly alphas in percent against FF5 + momentum, with Newey–West t-statistics (12 lags, scores kept on the monthly calendar). *Long leg* is the long portfolio's excess return, *short leg* the risk-free rate minus the short portfolio's return, and the spread is long minus short. Annualized figures compound the monthly alpha. **Only the 2010–2022 rows are out of sample.**

In [ ]:
def estimate_alpha(series, lags=newey_west_lags):
    sample = pd.concat([series.rename("y"), monthly_factors[factor_columns]], axis=1, join="inner")
    calendar = pd.period_range(sample.index.min(), sample.index.max(), freq="M")
    sample = sample.reindex(calendar)
    observed = sample.notna().all(axis=1).to_numpy()
    outcome = sample.loc[observed, "y"].to_numpy()
    design = np.column_stack([np.ones(observed.sum()), sample.loc[observed, factor_columns]])
    coefficients = np.linalg.lstsq(design, outcome, rcond=None)[0]
    n_obs, n_parameters = design.shape
    scores = np.zeros((len(calendar), n_parameters))
    scores[observed] = design * (outcome - design @ coefficients)[:, None]
    score_cov = scores.T @ scores
    for lag in range(1, min(lags, len(calendar) - 1) + 1):
        cross = scores[lag:].T @ scores[:-lag]
        score_cov += (1 - lag / (lags + 1)) * (cross + cross.T)
    bread = np.linalg.inv(design.T @ design)
    covariance = bread @ score_cov @ bread * n_obs / (n_obs - n_parameters)
    alpha, se = coefficients[0], np.sqrt(covariance[0, 0])
    return {"months": n_obs, "alpha_monthly_pct": 100 * alpha, "t_stat": alpha / se,
            "alpha_annual_pct": 100 * ((1 + alpha) ** 12 - 1), "market_beta": coefficients[1],
            "mean_monthly_pct": 100 * outcome.mean()}


def alpha_table(frame, columns, period_names):
    rows = []
    for column in columns:
        for period in period_names:
            series = frame.loc[frame.index.isin(periods[period]), column].dropna()
            rows.append({"strategy": column, "period": period, **estimate_alpha(series)})
    return pd.DataFrame(rows).set_index(["strategy", "period"])


strategies = ["Long leg", "Short leg", "Long-short spread"]
gross_table = alpha_table(main_frame, strategies, list(periods))
log_specifications("Gross alphas, main portfolio", gross_table["t_stat"])
display(gross_table.round(3))

In [ ]:
net_columns = [f"{strategy} | {scenario}" for scenario in cost_scenarios for strategy in strategies] + [
    f"{strategy} | estimated spread + {fee:.1%} borrow" for fee in (0.005, 0.01, 0.02) for strategy in ("Short leg", "Long-short spread")]
net_table = alpha_table(main_frame, net_columns, [headline, "1976–2009 (in sample)"])
log_specifications("Net alphas, main portfolio", net_table["t_stat"])
display(net_table[["months", "alpha_monthly_pct", "t_stat", "alpha_annual_pct"]].unstack("period").round(3))

## 9 · Where does the return come from?

These tables split the out-of-sample holdings by what happened **afterwards**. They describe the result and cannot be traded. The return is market-adjusted (stock return minus the CRSP value-weighted return) and signed for the leg, so a positive number helps the strategy. *Contribution* is the group's part of the leg's mean monthly market-adjusted return, in percentage points; the groups add up to the total row.

In [ ]:
def outcome_table(positions, leg):
    sample = positions.loc[positions["month"].isin(test_months)].join(market.rename("market"), on="month")
    sign = 1 if leg == "long" else -1
    sample["signed_adjusted"] = sign * (sample["ret"] - sample["market"])
    event_word = "added" if leg == "long" else "deleted (discretionary)"
    sample["outcome"] = np.select(
        [sample["exits"] & sample["y"].eq(1), sample["exits"],
         sample["y"].eq(1), sample["delists"]],
        [f"{event_word}: index trade in the holding month",
         "other index event in the holding month" if leg == "long" else "deleted after a merger: index trade in the holding month",
         f"{event_word} 1–2 months after the holding month", "delisted in the holding month"],
        default="no event within 3 months")
    sample.loc[sample["y"].isna() & sample["outcome"].eq("no event within 3 months"), "outcome"] = "label not complete"
    sample["weighted"] = sample["weight"] * sample["signed_adjusted"]
    months = sample["month"].nunique()
    table = sample.groupby("outcome").agg(
        position_months=("permno", "size"), mean_signed_adjusted_pct=("signed_adjusted", lambda r: 100 * r.mean()),
        contribution_to_monthly_mean_pct=("weighted", lambda r: 100 * r.sum() / months))
    table.loc["total"] = [len(sample), 100 * sample["signed_adjusted"].mean(), 100 * sample["weighted"].sum() / months]
    return table.sort_values("position_months", ascending=False)


for leg in ("long", "short"):
    print(f"{leg} leg, 2010–2022")
    display(outcome_table(main_details[leg]["positions"], leg).round(3))

## 10 · Robustness and benchmarks

All rows use the 2010–2022 out-of-sample period and gross returns unless stated. *No exit rule* holds every position for the full month, to show what the trade-date exit adds. *Size only* replaces the models with the 20 largest candidates (long) and the 20 smallest members (short), to show what the models add beyond size. These comparisons were planned with the design. A missing-return bracket is added only if some position-month has no return.

In [ ]:
robustness_frames = {
    "main (top 20, equal weight, exit at trade date)": main_frame,
    "no exit rule": build_strategy(variants["main"], exit_rule=False)[0],
    "top 10": build_strategy(variants["top 10"])[0],
    "top 40": build_strategy(variants["top 40"])[0],
    "value weight": build_strategy(variants["main"], weighting="value")[0],
    "size only (no model)": build_strategy(variants["size only"])[0],
}
rows = []
for name, frame in robustness_frames.items():
    for column in [*strategies, "Long-short spread | estimated spread"]:
        rows.append({"variant": name, "strategy": column,
                     **estimate_alpha(frame.loc[frame.index.isin(test_months), column].dropna())})
robustness_table = pd.DataFrame(rows).set_index(["variant", "strategy"])
log_specifications("Robustness, 2010–2022", robustness_table.loc[robustness_table.index.get_level_values(0) != list(robustness_frames)[0], "t_stat"])
display(robustness_table[["alpha_monthly_pct", "t_stat"]].unstack("strategy").swaplevel(0, 1, axis=1)
        .reindex(columns=[*strategies, "Long-short spread | estimated spread"], level=0).round(3))

missing = {leg: int(d["positions"]["missing_return"].sum()) for leg, d in main_details.items()}
if sum(missing.values()):
    rows = []
    for value in (-0.5, 0.5):
        frame = main_frame.copy()
        for leg in ("long", "short"):
            positions = main_details[leg]["positions"]
            shift = (positions["weight"] * value).where(positions["missing_return"], 0.0).groupby(positions["month"]).sum()
            frame[f"{leg}_ret"] = frame[f"{leg}_ret"] + shift.reindex(frame.index).fillna(0.0)
        frame["Long-short spread"] = frame["long_ret"] - frame["short_ret"]
        for period in [headline, "1976–2009 (in sample)"]:
            rows.append({"missing returns valued at": f"{value:+.0%}", "period": period,
                         **estimate_alpha(frame.loc[frame.index.isin(periods[period]), "Long-short spread"].dropna())})
    print("Position-months without a return:", missing)
    bracket = pd.DataFrame(rows).set_index(["missing returns valued at", "period"])
    log_specifications("Missing-return bracket", bracket["t_stat"])
    display(bracket[["alpha_monthly_pct", "t_stat"]].round(3))

In [ ]:
spec_table = pd.DataFrame(specification_log)
display(spec_table)
display(spec_table[["estimates", "|t| > 2", "|t| > 3"]].sum().rename("total"))

## Summary

**Prediction works out of sample.** For formation months December 2009 to September 2022, precision at 20 is 7.3% for additions (maximum attainable 18.6%, base rate 0.11%, AUC 0.987) and 10.7% for discretionary deletions (maximum attainable 12.3%, base rate 0.49%, AUC 0.967). The deletion list catches 88% of the deletions it could catch.

**Returns do not follow.** All figures are 2010–2022, out of sample, monthly alpha against FF5 + momentum:

| Strategy | Gross | Net, estimated spread | Net, 0.5% per side | Net, estimated spread + 2% borrow |
|---|---|---|---|---|
| Long leg | 0.06% (t 0.30) | 0.06% (t 0.26) | −0.13% (t −0.59) | – |
| Short leg | −0.19% (t −0.71) | −0.20% (t −0.75) | −0.39% (t −1.45) | −0.37% (t −1.38) |
| Long-short spread | −0.12% (t −0.31) | −0.14% (t −0.36) | −0.51% (t −1.28) | −0.31% (t −0.78) |

**Why.** Section 9 shows that the event itself still pays: true additions gain 3.5% against the market in the month of their index trade, and true deletions lose about 4.9% a month in the one to two months before theirs. But about 90% of the selected position-months have no event within three months. On the long side these are close to zero (−0.09% a month). On the short side they rebound by 0.84% a month against the position, which costs more than the true deletions earn. The deletion candidates are small members after large losses, and their typical rebound dominates the selling pressure of the few that are actually deleted.

**The exit rule adds nothing after 2009.** Its gross spread alpha is −0.12% with the exit and −0.10% without it. Section 1 explains why: in the training period most of the pressure fell in the last five sessions and on the trade date, but in 2010–2022 those windows are small (additions +0.6%, deletions −1.4%), and the move happens weeks to months earlier.

**Robustness.** No variant is significant in 2010–2022 (all |t| < 1.4): top 10, top 40, value weighting, no exit rule, and the size-only benchmark. The models do not beat the size-only lists in returns.

**In sample (1976–2009)**, the long leg earns 0.45% a month (t 2.44), concentrated in 2000–2009. The spread earns 0.35% (t 1.08), and it is negative after estimated costs. Quotes are sparse before the 1990s, so those costs often use the 0.5% fallback.

**Specifications.** 123 estimates with a t-statistic are reported in this notebook, 37 with |t| > 2. 31 of those are CAR sub-windows, which describe the event and are not strategy returns. In 2010–2022 the only strategy alphas with |t| > 2 are two net results that are significantly *negative* (spread and short leg at 1.0% per side).

**Conclusion.** With stocks only, the pre-event index effect cannot be harvested by predicting the event. The forecasts are accurate in relative terms, but the precision that is attainable (about one in ten names) is too low for the event return to outweigh the return of the names that are not affected, and the short side is hurt by the reversal of recent losers. This is a null result for the strategy as designed, stated before any further variation was tried.